# Notebook 01 -- Data Pipeline & Baseline RAG

**Config 1 (Baseline):** multilingual-e5-large + BM25 + RRF fusion + Qwen2.5-7B-Instruct (4-bit)

This notebook covers the full pipeline from data download to evaluation:
1. Setup & environment
2. Download & preprocess datasets
3. Legal-aware chunking
4. Build FAISS & BM25 indexes
5. Baseline RAG pipeline
6. Evaluation on gold test set
7. Results summary

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q -r /content/c493/requirements.txt 2>/dev/null || \
    !pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/berkay-aktas/c493.git"
PROJECT_ROOT = Path("/content/c493")

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print(f"Repo already cloned at {PROJECT_ROOT}")
    !cd {PROJECT_ROOT} && git pull

os.chdir(PROJECT_ROOT)

In [ ]:
import sys
import logging

# Add project root to path so src.* imports work
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
from src.utils.config import load_config, set_seeds, get_device, free_gpu_memory
from src.data.preprocessor import download_datasets, merge_and_dedup, save_corpus, load_corpus
from src.data.chunker import chunk_corpus
from src.data.gold_set import load_gold_set, gold_set_stats
from src.retrieval.dense import build_faiss_index, load_faiss_index, dense_search, load_embedding_model
from src.retrieval.bm25 import build_bm25_index, load_bm25_index, bm25_search
from src.retrieval.fusion import rrf_merge
from src.evaluation.metrics import retrieval_metrics, generation_metrics

cfg = load_config()
set_seeds(cfg)
device = get_device()

print(f"Device: {device}")
print(f"Colab: {cfg['_is_colab']}")
print(f"Drive root: {cfg['paths']['drive_root']}")

In [ ]:
# Ensure Drive output directories exist
DRIVE_ROOT = Path(cfg["paths"]["drive_root"])
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "indexes").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "results").mkdir(parents=True, exist_ok=True)

print(f"Drive output: {DRIVE_ROOT}")

## 2. Download & Preprocess Data

In [ ]:
RAW_DIR = DRIVE_ROOT / "data" / "raw"
CORPUS_PATH = Path(cfg["paths"]["corpus_parquet"])

if CORPUS_PATH.exists():
    print(f"Corpus already exists at {CORPUS_PATH}, skipping download.")
    corpus_df = load_corpus(CORPUS_PATH)
else:
    raw_paths = download_datasets(cfg, output_dir=RAW_DIR)
    print(f"Downloaded datasets: {list(raw_paths.keys())}")

    corpus_df = merge_and_dedup(raw_paths)
    save_corpus(corpus_df, CORPUS_PATH)

print(f"\n--- Corpus Statistics ---")
print(f"Total documents: {len(corpus_df):,}")
print(f"Columns: {list(corpus_df.columns)}")
print(f"Memory usage: {corpus_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

if "_source" in corpus_df.columns:
    print(f"\nPer-source breakdown:")
    print(corpus_df["_source"].value_counts().to_string())

corpus_df.head(3)

## 3. Chunk Corpus

In [ ]:
CHUNKS_PATH = Path(cfg["paths"]["chunks_parquet"])

if CHUNKS_PATH.exists():
    print(f"Chunks already exist at {CHUNKS_PATH}, loading from disk.")
    import pandas as pd
    chunks_df = pd.read_parquet(CHUNKS_PATH)
else:
    corpus_df = load_corpus(CORPUS_PATH)

    chunks_df = chunk_corpus(
        corpus_df,
        text_column="text",
        max_tokens=cfg["chunking"]["max_tokens"],
        overlap_tokens=cfg["chunking"]["overlap_tokens"],
    )

    CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
    chunks_df.to_parquet(CHUNKS_PATH, index=False)
    print(f"Saved {len(chunks_df):,} chunks to {CHUNKS_PATH}")

# Stats
chunks_df["_word_count"] = chunks_df["text"].str.split().str.len()

print(f"\n--- Chunk Distribution ---")
print(f"Total chunks: {len(chunks_df):,}")
print(f"Avg words/chunk: {chunks_df['_word_count'].mean():.1f}")
print(f"Median words/chunk: {chunks_df['_word_count'].median():.1f}")
print(f"Min / Max: {chunks_df['_word_count'].min()} / {chunks_df['_word_count'].max()}")

if "_source" in chunks_df.columns:
    print(f"\nChunks per source:")
    print(chunks_df["_source"].value_counts().to_string())

chunks_df = chunks_df.drop(columns=["_word_count"])
chunks_df.head(3)

## 4. Build Indexes

In [ ]:
import pandas as pd

FAISS_PATH = Path(cfg["paths"]["faiss_index"])
BM25_PATH = Path(cfg["paths"]["bm25_index"])

# Load chunks as list of dicts for index builders
chunks_df = pd.read_parquet(CHUNKS_PATH)
chunk_records = chunks_df.to_dict(orient="records")

In [ ]:
# Build FAISS index
if FAISS_PATH.exists():
    print(f"FAISS index exists at {FAISS_PATH}, loading...")
    faiss_index, faiss_mapping = load_faiss_index(FAISS_PATH)
else:
    faiss_index, faiss_mapping = build_faiss_index(
        chunks=chunk_records,
        model_name=cfg["models"]["embedding"],
        save_path=FAISS_PATH,
        batch_size=64,
        nlist=cfg["retrieval"]["faiss"]["nlist"],
        m=cfg["retrieval"]["faiss"]["m"],
        nbits=cfg["retrieval"]["faiss"]["nbits"],
    )

print(f"FAISS index: {faiss_index.ntotal:,} vectors")

In [ ]:
# Build BM25 index
if BM25_PATH.exists():
    print(f"BM25 index exists at {BM25_PATH}, loading...")
    bm25_index, bm25_mapping = load_bm25_index(BM25_PATH)
else:
    bm25_index, bm25_mapping = build_bm25_index(
        chunks=chunk_records,
        save_path=BM25_PATH,
    )

print(f"BM25 index: {len(bm25_mapping):,} chunks")

In [ ]:
# Sanity check: search a sample query on both indexes
SAMPLE_QUERY = "Kasten adam oldurme sucunun cezasi nedir?"

# Load embedding model for dense search
embed_model = load_embedding_model(cfg["models"]["embedding"])

dense_results = dense_search(
    query=SAMPLE_QUERY,
    index=faiss_index,
    chunk_mapping=faiss_mapping,
    model=embed_model,
    k=3,
    nprobe=cfg["retrieval"]["faiss"]["nprobe"],
)

bm25_results = bm25_search(
    query=SAMPLE_QUERY,
    index=bm25_index,
    chunk_mapping=bm25_mapping,
    k=3,
)

print("=== Dense (FAISS) Top-3 ===")
for i, r in enumerate(dense_results, 1):
    print(f"\n[{i}] score={r.score:.4f}")
    print(f"    {r.text[:200]}...")

print("\n=== BM25 Top-3 ===")
for i, r in enumerate(bm25_results, 1):
    print(f"\n[{i}] score={r.score:.4f}")
    print(f"    {r.text[:200]}...")

## 5. Baseline RAG Pipeline

In [ ]:
# Free GPU memory from embedding model before loading LLM
del embed_model
free_gpu_memory()
print("GPU memory freed.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_NAME = cfg["models"]["llm"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_NAME} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
SYSTEM_PROMPT = (
    "Sen bir Turk hukuku uzmanisn. Sana verilen baglam paragraflarini kullanarak "
    "soruyu yanitla. Yanitinda ilgili kanun maddelerine atifta bulun. "
    "Eger baglam bilgisi yeterli degilse, bunu acikca belirt ve "
    "bilmedigin konularda uydurma yapma."
)


def format_context(results, top_k=10):
    """Format retrieval results into a numbered context string."""
    parts = []
    for i, r in enumerate(results[:top_k], 1):
        parts.append(f"[{i}] {r.text}")
    return "\n\n".join(parts)


def build_prompt(question, context_str):
    """Build the chat-formatted prompt for the LLM."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Baglam:\n{context_str}\n\n"
                f"Soru: {question}\n\n"
                "Lutfen yukaridaki baglami kullanarak soruyu yanitla. "
                "Hangi kaynaklardan ([1], [2], ...) yararlandigini belirt."
            ),
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.inference_mode()
def generate_answer(prompt, max_new_tokens=None, temperature=None, top_p=None):
    """Generate an answer from the LLM given a formatted prompt."""
    max_new_tokens = max_new_tokens or cfg["generation"]["max_new_tokens"]
    temperature = temperature or cfg["generation"]["temperature"]
    top_p = top_p or cfg["generation"]["top_p"]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
def baseline_rag(question, embed_model, faiss_index, faiss_mapping,
                 bm25_index, bm25_mapping, cfg):
    """Run the full baseline RAG pipeline for a single question.

    Steps:
      1. Dense search (FAISS)
      2. BM25 search
      3. RRF merge
      4. Format top-10 passages into prompt
      5. Generate answer with citations

    Returns:
        Tuple of (answer_text, merged_results).
    """
    # 1. Dense retrieval
    d_results = dense_search(
        query=question,
        index=faiss_index,
        chunk_mapping=faiss_mapping,
        model=embed_model,
        k=cfg["retrieval"]["dense_top_k"],
        nprobe=cfg["retrieval"]["faiss"]["nprobe"],
    )

    # 2. BM25 retrieval
    b_results = bm25_search(
        query=question,
        index=bm25_index,
        chunk_mapping=bm25_mapping,
        k=cfg["retrieval"]["bm25_top_k"],
    )

    # 3. RRF fusion
    merged = rrf_merge(
        d_results, b_results,
        k=cfg["retrieval"]["fusion"]["rrf_k"],
        top_k=cfg["retrieval"]["final_top_k"],
    )

    # 4. Build prompt
    context_str = format_context(merged, top_k=cfg["retrieval"]["final_top_k"])
    prompt = build_prompt(question, context_str)

    # 5. Generate
    answer = generate_answer(prompt)

    return answer, merged

In [ ]:
# Reload embedding model for retrieval inside the pipeline
embed_model = load_embedding_model(cfg["models"]["embedding"])

In [ ]:
# Test with example questions
TEST_QUESTIONS = [
    "Kasten adam oldurme sucunun cezasi nedir?",
    "Kira sozlesmesinde kiracinin haklari nelerdir?",
    "Idari yargida dava acma suresi ne kadardir?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'='*80}")
    print(f"SORU: {q}")
    print("=" * 80)
    answer, results = baseline_rag(
        q, embed_model, faiss_index, faiss_mapping,
        bm25_index, bm25_mapping, cfg,
    )
    print(f"\nYANIT:\n{answer}")
    print(f"\nRetrieved {len(results)} passages (showing top-3 IDs):")
    for r in results[:3]:
        print(f"  - {r.chunk_id} (score={r.score:.4f})")

## 6. Evaluate Baseline

In [ ]:
from tqdm.auto import tqdm

# Load gold test set
GOLD_PATH = Path(cfg["evaluation"]["gold_set_path"])
if not GOLD_PATH.is_absolute():
    GOLD_PATH = PROJECT_ROOT / GOLD_PATH

# Fall back to template if full set not yet created
if not GOLD_PATH.exists():
    GOLD_PATH = PROJECT_ROOT / "data" / "gold" / "gold_test_template.json"
    print(f"Full gold set not found, using template: {GOLD_PATH}")

gold_data = load_gold_set(GOLD_PATH)
stats = gold_set_stats(gold_data)
print(f"Gold set: {stats['total']} questions")
print(f"  Domains: {stats['by_domain']}")
print(f"  Difficulties: {stats['by_difficulty']}")
print(f"  Answerable: {stats['answerable']}, Unanswerable: {stats['unanswerable']}")

In [ ]:
# Run all gold questions through the pipeline
predictions = []
references = []
all_retrieved_ids = []
all_relevant_ids = []

for item in tqdm(gold_data, desc="Evaluating"):
    question = item["question"]
    gold_answer = item["gold_answer"]
    relevant_ids = item.get("relevant_doc_ids", [])

    answer, results = baseline_rag(
        question, embed_model, faiss_index, faiss_mapping,
        bm25_index, bm25_mapping, cfg,
    )

    predictions.append(answer)
    references.append(gold_answer)
    all_retrieved_ids.append([r.chunk_id for r in results])
    all_relevant_ids.append(relevant_ids)

print(f"Evaluated {len(predictions)} questions.")

In [ ]:
# Compute retrieval metrics (only for questions with relevant_doc_ids)
has_relevant = [
    i for i, ids in enumerate(all_relevant_ids) if len(ids) > 0
]

if has_relevant:
    filtered_retrieved = [all_retrieved_ids[i] for i in has_relevant]
    filtered_relevant = [all_relevant_ids[i] for i in has_relevant]
    ret_metrics = retrieval_metrics(filtered_retrieved, filtered_relevant)
else:
    print("No relevant_doc_ids in gold set -- retrieval metrics use empty baselines.")
    print("Populate relevant_doc_ids after indexing to get meaningful retrieval scores.")
    ret_metrics = {
        "recall@5": None,
        "recall@10": None,
        "mrr": None,
        "ndcg@10": None,
    }

print("\n--- Retrieval Metrics ---")
for k, v in ret_metrics.items():
    print(f"  {k}: {v if v is None else f'{v:.4f}'}")

In [ ]:
# Compute generation metrics
gen_metrics = generation_metrics(predictions, references)

print("\n--- Generation Metrics ---")
for k, v in gen_metrics.items():
    print(f"  {k}: {v:.4f}")

## 7. Results Summary

In [ ]:
import json
import pandas as pd

# Combine all metrics
baseline_results = {
    "config": "Config 1 -- Baseline",
    "embedding_model": cfg["models"]["embedding"],
    "llm_model": cfg["models"]["llm"],
    "quantization": "4-bit NF4",
    "retrieval": {
        "method": "Dense (FAISS IVF-PQ) + BM25 + RRF",
        "dense_top_k": cfg["retrieval"]["dense_top_k"],
        "bm25_top_k": cfg["retrieval"]["bm25_top_k"],
        "final_top_k": cfg["retrieval"]["final_top_k"],
        "rrf_k": cfg["retrieval"]["fusion"]["rrf_k"],
    },
    "metrics": {
        "retrieval": {k: float(v) if v is not None else None for k, v in ret_metrics.items()},
        "generation": {k: float(v) for k, v in gen_metrics.items()},
    },
    "gold_set_size": len(gold_data),
}

# Display as table
display_rows = []
for category in ["retrieval", "generation"]:
    for metric, value in baseline_results["metrics"][category].items():
        display_rows.append({
            "Category": category.capitalize(),
            "Metric": metric,
            "Score": f"{value:.4f}" if value is not None else "N/A",
        })

results_table = pd.DataFrame(display_rows)
print("\n" + "=" * 50)
print("  CONFIG 1 (BASELINE) -- RESULTS")
print("=" * 50)
print(results_table.to_string(index=False))
print("=" * 50)

In [ ]:
# Save results to Drive
results_path = DRIVE_ROOT / "results" / "baseline_config1.json"
results_path.parent.mkdir(parents=True, exist_ok=True)

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print(f"Results saved to {results_path}")

In [ ]:
# Cleanup
del model, tokenizer
free_gpu_memory()
print("GPU memory freed. Notebook complete.")